# Calculate mean and standard deviation for language model annotations

In [ ]:
import json
import numpy as np
from pathlib import Path


def calculate_metrics_across_runs(
    base_path="../training_data",
    folder_names=["qwen_0.6B_inst", "qwen_0.6B_inst_2", "qwen_0.6B_inst_3"],
    output_filename="aggregated_results.json",
):
    """
    Calculate mean and standard deviation for all metrics across multiple experiment runs.

    Args:
        base_path (str): Base path to the training data directory
        folder_names (list): List of folder names containing the result files
        output_filename (str): Name of the output file to save aggregated results

    Returns:
        dict: Dictionary containing mean and std for all metrics
    """

    # Initialize storage for all metrics
    all_results = []

    # Read all result files
    for folder_name in folder_names:
        file_path = (
            Path(base_path) / folder_name / "instruction_tuning_test_results.json"
        )

        try:
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)
                all_results.append(data)
                print(f"Successfully loaded: {file_path}")
        except FileNotFoundError:
            print(f"Warning: File not found: {file_path}")
            continue
        except json.JSONDecodeError as e:
            print(f"Error reading JSON from {file_path}: {e}")
            continue

    if len(all_results) < 2:
        print("Error: Need at least 2 valid result files to calculate statistics")
        return None

    print(f"Loaded {len(all_results)} result files")

    # Initialize aggregated results structure
    aggregated = {
        "metadata": {
            "num_runs": len(all_results),
            "source_folders": folder_names[: len(all_results)],
            "aggregation_method": "mean_and_std",
        },
        "results": {},
        "timing": {},
    }

    # Copy model_info and generation_params from first file
    if "model_info" in all_results[0]:
        aggregated["model_info"] = all_results[0]["model_info"]
    if "generation_params" in all_results[0]:
        aggregated["generation_params"] = all_results[0]["generation_params"]

    # Process results section
    thinking_modes = ["with_thinking", "without_thinking"]
    datasets = ["ynacc", "iac", "reddit"]
    metrics = [
        "accuracy",
        "precision",
        "recall",
        "f1",
        "tp",
        "fp",
        "tn",
        "fn",
        "total_samples",
    ]

    for thinking_mode in thinking_modes:
        aggregated["results"][thinking_mode] = {}

        for dataset in datasets:
            aggregated["results"][thinking_mode][dataset] = {"instruction_tuning": {}}

            for metric in metrics:
                values = []

                # Collect values from all runs
                for result in all_results:
                    try:
                        value = result["results"][thinking_mode][dataset][
                            "instruction_tuning"
                        ][metric]
                        values.append(value)
                    except KeyError:
                        print(
                            f"Warning: Missing {thinking_mode}/{dataset}/{metric} in one of the files"
                        )
                        continue

                if values:
                    # Calculate mean and std
                    mean_val = np.mean(values)
                    std_val = np.std(values, ddof=1) if len(values) > 1 else 0.0

                    aggregated["results"][thinking_mode][dataset]["instruction_tuning"][
                        metric
                    ] = {
                        "mean": float(mean_val),
                        "std": float(std_val),
                        "values": values,
                        "count": len(values),
                    }

    # Process timing section
    timing_categories = ["with_thinking", "without_thinking", "overall"]
    timing_metrics = ["duration_seconds", "duration_minutes", "samples_per_second"]

    for category in timing_categories:
        if category not in aggregated["timing"]:
            aggregated["timing"][category] = {}

        if category == "overall":
            # Handle overall timing
            for metric in ["duration_seconds", "duration_minutes", "duration_hours"]:
                values = []
                for result in all_results:
                    try:
                        value = result["timing"]["overall"][metric]
                        values.append(value)
                    except KeyError:
                        continue

                if values:
                    mean_val = np.mean(values)
                    std_val = np.std(values, ddof=1) if len(values) > 1 else 0.0

                    aggregated["timing"][category][metric] = {
                        "mean": float(mean_val),
                        "std": float(std_val),
                        "values": values,
                        "count": len(values),
                    }

            # Handle total duration
            total_values = []
            for result in all_results:
                try:
                    value = result["timing"][category]["duration_seconds"]
                    total_values.append(value)
                except KeyError:
                    continue

            if total_values:
                aggregated["timing"][category]["total"] = {
                    "mean": float(np.mean(total_values)),
                    "std": (
                        float(np.std(total_values, ddof=1))
                        if len(total_values) > 1
                        else 0.0
                    ),
                    "values": total_values,
                    "count": len(total_values),
                }
        else:
            # Handle with_thinking and without_thinking timing
            for dataset in datasets:
                if dataset not in aggregated["timing"][category]:
                    aggregated["timing"][category][dataset] = {}

                for metric in timing_metrics:
                    values = []
                    for result in all_results:
                        try:
                            value = result["timing"][category][dataset][metric]
                            values.append(value)
                        except KeyError:
                            continue

                    if values:
                        mean_val = np.mean(values)
                        std_val = np.std(values, ddof=1) if len(values) > 1 else 0.0

                        aggregated["timing"][category][dataset][metric] = {
                            "mean": float(mean_val),
                            "std": float(std_val),
                            "values": values,
                            "count": len(values),
                        }

            # Handle total for each thinking mode
            total_duration_values = []
            total_duration_minutes_values = []

            for result in all_results:
                try:
                    total_sec = result["timing"][category]["total"]["duration_seconds"]
                    total_min = result["timing"][category]["total"]["duration_minutes"]
                    total_duration_values.append(total_sec)
                    total_duration_minutes_values.append(total_min)
                except KeyError:
                    continue

            if total_duration_values:
                aggregated["timing"][category]["total"] = {
                    "duration_seconds": {
                        "mean": float(np.mean(total_duration_values)),
                        "std": (
                            float(np.std(total_duration_values, ddof=1))
                            if len(total_duration_values) > 1
                            else 0.0
                        ),
                        "values": total_duration_values,
                        "count": len(total_duration_values),
                    },
                    "duration_minutes": {
                        "mean": float(np.mean(total_duration_minutes_values)),
                        "std": (
                            float(np.std(total_duration_minutes_values, ddof=1))
                            if len(total_duration_minutes_values) > 1
                            else 0.0
                        ),
                        "values": total_duration_minutes_values,
                        "count": len(total_duration_minutes_values),
                    },
                }

    # Save aggregated results
    output_path = Path(base_path) / output_filename
    try:
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(aggregated, f, indent=2, ensure_ascii=False)
        print(f"Aggregated results saved to: {output_path}")
    except Exception as e:
        print(f"Error saving aggregated results: {e}")

    return aggregated


def print_summary_stats(aggregated_results):
    """
    Print a summary of the key performance metrics.

    Args:
        aggregated_results (dict): The aggregated results dictionary
    """
    if not aggregated_results:
        print("No results to summarize")
        return

    print("=" * 80)
    print("PERFORMANCE METRICS SUMMARY")
    print("=" * 80)

    thinking_modes = ["with_thinking", "without_thinking"]
    datasets = ["ynacc", "iac", "reddit"]
    key_metrics = ["accuracy", "precision", "recall", "f1"]

    for thinking_mode in thinking_modes:
        print(f"\n{thinking_mode.upper().replace('_', ' ')} MODE:")
        print("-" * 50)

        for dataset in datasets:
            print(f"\n  {dataset.upper()} Dataset:")

            for metric in key_metrics:
                try:
                    data = aggregated_results["results"][thinking_mode][dataset][
                        "instruction_tuning"
                    ][metric]
                    mean_val = data["mean"]
                    std_val = data["std"]
                    print(
                        f"    {metric.capitalize():10s}: {mean_val:.3f} ± {std_val:.3f}"
                    )
                except KeyError:
                    print(f"    {metric.capitalize():10s}: Missing data")

    print("\n" + "=" * 80)

In [ ]:
# Calculate aggregated metrics across all three runs
aggregated_results = calculate_metrics_across_runs(
    base_path="../training_data",
    folder_names=["qwen_0.6B_inst", "qwen_0.6B_inst_2", "qwen_0.6B_inst_3"],
    output_filename="qwen_0.6B_inst_aggregated_results.json",
)

# Print summary statistics
if aggregated_results:
    print_summary_stats(aggregated_results)

Successfully loaded: ../training_data/qwen_0.6B_inst/instruction_tuning_test_results.json
Successfully loaded: ../training_data/qwen_0.6B_inst_2/instruction_tuning_test_results.json
Successfully loaded: ../training_data/qwen_0.6B_inst_3/instruction_tuning_test_results.json
Loaded 3 result files
Aggregated results saved to: ../training_data/qwen_0.6B_inst_aggregated_results.json
PERFORMANCE METRICS SUMMARY

WITH THINKING MODE:
--------------------------------------------------

  YNACC Dataset:
    Accuracy  : 0.617 ± 0.023
    Precision : 0.604 ± 0.015
    Recall    : 0.719 ± 0.045
    F1        : 0.656 ± 0.028

  IAC Dataset:
    Accuracy  : 0.703 ± 0.050
    Precision : 0.825 ± 0.033
    Recall    : 0.793 ± 0.032
    F1        : 0.809 ± 0.032

  REDDIT Dataset:
    Accuracy  : 0.713 ± 0.015
    Precision : 0.777 ± 0.006
    Recall    : 0.836 ± 0.033
    F1        : 0.805 ± 0.014

WITHOUT THINKING MODE:
--------------------------------------------------

  YNACC Dataset:
    Accuracy 

In [ ]:
# Calculate aggregated metrics across all three runs
aggregated_results = calculate_metrics_across_runs(
    base_path="../training_data",
    folder_names=[
        "qwen_0.6B_inst_few_shot",
        "qwen_0.6B_inst_few_shot_2",
        "qwen_0.6B_inst_few_shot_3",
    ],
    output_filename="qwen_0.6B_inst_few_shot/aggregated_results.json",
)

# Print summary statistics
if aggregated_results:
    print_summary_stats(aggregated_results)

Successfully loaded: ../training_data/qwen_0.6B_inst_few_shot/instruction_tuning_test_results.json
Successfully loaded: ../training_data/qwen_0.6B_inst_few_shot_2/instruction_tuning_test_results.json
Successfully loaded: ../training_data/qwen_0.6B_inst_few_shot_3/instruction_tuning_test_results.json
Loaded 3 result files
Aggregated results saved to: ../training_data/qwen_0.6B_inst_few_shot/aggregated_results.json
PERFORMANCE METRICS SUMMARY

WITH THINKING MODE:
--------------------------------------------------

  YNACC Dataset:
    Accuracy  : 0.627 ± 0.025
    Precision : 0.643 ± 0.019
    Recall    : 0.601 ± 0.049
    F1        : 0.621 ± 0.035

  IAC Dataset:
    Accuracy  : 0.673 ± 0.055
    Precision : 0.858 ± 0.015
    Recall    : 0.705 ± 0.102
    F1        : 0.770 ± 0.055

  REDDIT Dataset:
    Accuracy  : 0.683 ± 0.040
    Precision : 0.808 ± 0.039
    Recall    : 0.728 ± 0.035
    F1        : 0.765 ± 0.030

WITHOUT THINKING MODE:
----------------------------------------------

In [ ]:
# Calculate aggregated metrics across all three runs
aggregated_results = calculate_metrics_across_runs(
    base_path="../training_data",
    folder_names=[
        "qwen_1.7B_inst",
        "qwen_1.7B_inst_2",
        "qwen_1.7B_inst_3",
    ],
    output_filename="qwen_1.7B_inst/aggregated_results.json",
)

# Print summary statistics
if aggregated_results:
    print_summary_stats(aggregated_results)

Successfully loaded: ../training_data/qwen_1.7B_inst/instruction_tuning_test_results.json
Successfully loaded: ../training_data/qwen_1.7B_inst_2/instruction_tuning_test_results.json
Successfully loaded: ../training_data/qwen_1.7B_inst_3/instruction_tuning_test_results.json
Loaded 3 result files
Aggregated results saved to: ../training_data/qwen_1.7B_inst/aggregated_results.json
PERFORMANCE METRICS SUMMARY

WITH THINKING MODE:
--------------------------------------------------

  YNACC Dataset:
    Accuracy  : 0.633 ± 0.015
    Precision : 0.787 ± 0.005
    Recall    : 0.386 ± 0.041
    F1        : 0.517 ± 0.036

  IAC Dataset:
    Accuracy  : 0.710 ± 0.053
    Precision : 0.899 ± 0.012
    Recall    : 0.713 ± 0.077
    F1        : 0.794 ± 0.047

  REDDIT Dataset:
    Accuracy  : 0.757 ± 0.031
    Precision : 0.840 ± 0.025
    Recall    : 0.812 ± 0.033
    F1        : 0.826 ± 0.023

WITHOUT THINKING MODE:
--------------------------------------------------

  YNACC Dataset:
    Accuracy 

# Explore correlations between constructiveness and other labels

In [ ]:
import pandas as pd

# Load the reddit_annotated.csv file
df_reddit_annotated = pd.read_csv("../training_data/reddit_annotated.csv")

print(f"Shape of the DataFrame: {df_reddit_annotated.shape}")
print(f"Number of rows: {len(df_reddit_annotated)}")
print(f"Number of columns: {len(df_reddit_annotated.columns)}")
print(f"Column names: {list(df_reddit_annotated.columns)}")
print("\nFirst few rows:")
print(df_reddit_annotated.head())
print("\nData types:")
print(df_reddit_annotated.dtypes)
print("\nUnique values in each column:")
for col in df_reddit_annotated.columns:
    unique_count = df_reddit_annotated[col].nunique()
    print(f"{col}: {unique_count} unique values")
    if unique_count < 10:  # Show actual values if there are few unique values
        print(f"  Values: {df_reddit_annotated[col].unique()}")

Shape of the DataFrame: (220, 5)
Number of rows: 220
Number of columns: 5
Column names: ['sdid', 'text', 'constructiveness', 'type', 'coherence']

First few rows:
      sdid                                               text  \
0  fpbdhj3  [author0] If you could turn any movie into a p...   
1  fpc96gr  [author0] Cute Sana \n\n[author1] When is this...   
2  fpap1xt  [author0] [LF] Park Bench DIY [FT] Bells Looki...   
3  fp855pn  [author0] [LF] Catalogue Party [FT] My Catalog...   
4  fpb9txe  [author0] The Justin Amash conundrum has an ea...   

  constructiveness                 type          coherence  
0     Constructive        Argumentative  5-Highly coherent  
1     Constructive                  NaN  5-Highly coherent  
2     Constructive  Positive/respectful  5-Highly coherent  
3     Constructive  Positive/respectful  5-Highly coherent  
4     Constructive        Argumentative  5-Highly coherent  

Data types:
sdid                object
text                object
constructiven

In [ ]:
# Analyze correlations between constructiveness and other categorical variables
import pandas as pd

print("=" * 80)
print("ANALYSIS: CONSTRUCTIVENESS vs TYPE and COHERENCE")
print("=" * 80)

# Group by constructiveness (including NaN as a valid category)
constructiveness_groups = df_reddit_annotated.groupby("constructiveness", dropna=False)

print(f"\nTotal samples: {len(df_reddit_annotated)}")
print(f"Constructiveness distribution:")
print(df_reddit_annotated["constructiveness"].value_counts(dropna=False))

# Analyze TYPE distribution within each constructiveness group
print("\n" + "=" * 60)
print("TYPE DISTRIBUTION BY CONSTRUCTIVENESS")
print("=" * 60)

for constructiveness_value, group in constructiveness_groups:
    print(f"\n--- {constructiveness_value} (n={len(group)}) ---")
    type_counts = group["type"].value_counts(dropna=False)
    type_percentages = group["type"].value_counts(normalize=True, dropna=False) * 100

    print("Type counts and percentages:")
    for type_val, count in type_counts.items():
        percentage = type_percentages[type_val]
        print(f"  {type_val}: {count} ({percentage:.1f}%)")

# Analyze COHERENCE distribution within each constructiveness group
print("\n" + "=" * 60)
print("COHERENCE DISTRIBUTION BY CONSTRUCTIVENESS")
print("=" * 60)

for constructiveness_value, group in constructiveness_groups:
    print(f"\n--- {constructiveness_value} (n={len(group)}) ---")
    coherence_counts = group["coherence"].value_counts(dropna=False)
    coherence_percentages = (
        group["coherence"].value_counts(normalize=True, dropna=False) * 100
    )

    print("Coherence counts and percentages:")
    for coherence_val, count in coherence_counts.items():
        percentage = coherence_percentages[coherence_val]
        print(f"  {coherence_val}: {count} ({percentage:.1f}%)")

ANALYSIS: CONSTRUCTIVENESS vs TYPE and COHERENCE

Total samples: 220
Constructiveness distribution:
constructiveness
Constructive        132
Not constructive     52
NaN                  36
Name: count, dtype: int64

TYPE DISTRIBUTION BY CONSTRUCTIVENESS

--- Constructive (n=132) ---
Type counts and percentages:
  Positive/respectful: 65 (49.2%)
  Argumentative: 21 (15.9%)
  nan: 9 (6.8%)
  Snarky/humorous: 8 (6.1%)
  Prersonal stories: 8 (6.1%)
  Prersonal stories, Positive/respectful: 7 (5.3%)
  Positive/respectful, Prersonal stories: 5 (3.8%)
  Argumentative, Positive/respectful: 2 (1.5%)
  Positive/respectful, Argumentative: 2 (1.5%)
  Off-Topic/digression, Prersonal stories: 1 (0.8%)
  Off-Topic/digression: 1 (0.8%)
  Snarky/humorous, Off-Topic/digression: 1 (0.8%)
  Positive/respectful, Snarky/humorous: 1 (0.8%)
  Argumentative, Flamewar: 1 (0.8%)

--- Not constructive (n=52) ---
Type counts and percentages:
  Flamewar: 10 (19.2%)
  Argumentative: 8 (15.4%)
  Snarky/humorous: 8 (1

In [ ]:
# Create normalized percentage tables for better pattern analysis
print("\n" + "=" * 80)
print("NORMALIZED PERCENTAGE ANALYSIS")
print("=" * 80)

# Percentage of each constructiveness category within each type
print("\nWithin each TYPE, what percentage is Constructive/Not constructive/Missing?")
type_constructiveness = (
    pd.crosstab(
        df_reddit_annotated["type"],
        df_reddit_annotated["constructiveness"],
        dropna=False,
        normalize="index",
    )
    * 100
)
print(type_constructiveness.round(1))


NORMALIZED PERCENTAGE ANALYSIS

Within each TYPE, what percentage is Constructive/Not constructive/Missing?
constructiveness                           Constructive  Not constructive  \
type                                                                        
Argumentative                                      67.7              25.8   
Argumentative, Flamewar                            16.7              83.3   
Argumentative, Positive/respectful                100.0               0.0   
Argumentative, Prersonal stories                    0.0             100.0   
Flamewar                                            0.0             100.0   
Off-Topic/digression                                9.1              63.6   
Off-Topic/digression, Positive/respectful           0.0             100.0   
Off-Topic/digression, Prersonal stories            50.0              50.0   
Off-Topic/digression, Snarky/humorous               0.0             100.0   
Positive/respectful                         

In [ ]:
# Enhanced analysis: Split multi-value type entries and count individually
import pandas as pd
from collections import Counter

print("\n" + "=" * 80)
print("ENHANCED ANALYSIS: INDIVIDUAL TYPE COMPONENTS")
print("=" * 80)


def split_type_values(type_series):
    """
    Split comma-separated type values and return individual components
    """
    individual_types = []
    for type_val in type_series:
        if pd.isna(type_val):
            individual_types.append("NaN")
        else:
            # Split by comma and strip whitespace
            components = [component.strip() for component in str(type_val).split(",")]
            individual_types.extend(components)
    return individual_types


# Analysis by constructiveness group with individual type counting
print("TYPE COMPONENTS DISTRIBUTION BY CONSTRUCTIVENESS")
print("=" * 60)

for constructiveness_value, group in constructiveness_groups:
    print(f"\n--- {constructiveness_value} (n={len(group)}) ---")

    # Get individual type components for this group
    individual_types = split_type_values(group["type"])
    type_counter = Counter(individual_types)

    # Calculate total occurrences (can be more than group size due to multi-value entries)
    total_type_mentions = sum(type_counter.values())

    print(f"Total type mentions: {total_type_mentions} (from {len(group)} threads)")
    print("Individual type component counts and percentages:")

    # Sort by count descending
    for type_component, count in type_counter.most_common():
        percentage = (count / total_type_mentions) * 100
        print(f"  {type_component}: {count} ({percentage:.1f}%)")

# Create a comprehensive view of all individual type components
print(f"\n{'='*80}")
print("OVERALL INDIVIDUAL TYPE COMPONENTS ANALYSIS")
print("=" * 80)

# Get all individual type components across the entire dataset
all_individual_types = split_type_values(df_reddit_annotated["type"])
all_type_counter = Counter(all_individual_types)
total_all_mentions = sum(all_type_counter.values())

print(f"Total type component mentions across all data: {total_all_mentions}")
print(f"Unique type components: {len(all_type_counter)}")
print("\nAll individual type components (sorted by frequency):")

for type_component, count in all_type_counter.most_common():
    percentage = (count / total_all_mentions) * 100
    print(f"  {type_component}: {count} ({percentage:.1f}%)")

# Add COHERENCE ANALYSIS
print(f"\n{'='*80}")
print("OVERALL COHERENCE ANALYSIS")
print("=" * 80)

# Overall coherence distribution
coherence_counts = df_reddit_annotated["coherence"].value_counts(dropna=False)
total_coherence_entries = len(df_reddit_annotated)

print(f"Total threads with coherence ratings: {total_coherence_entries}")
print(f"Unique coherence levels: {df_reddit_annotated['coherence'].nunique()}")
print("\nOverall coherence distribution:")

for coherence_level, count in coherence_counts.items():
    percentage = (count / total_coherence_entries) * 100
    print(f"  {coherence_level}: {count} ({percentage:.1f}%)")

# COHERENCE DISTRIBUTION BY CONSTRUCTIVENESS
print(f"\n{'='*80}")
print("COHERENCE DISTRIBUTION BY CONSTRUCTIVENESS")
print("=" * 80)

for constructiveness_value, group in constructiveness_groups:
    print(f"\n--- {constructiveness_value} (n={len(group)}) ---")

    coherence_counts_group = group["coherence"].value_counts(dropna=False)

    print("Coherence level counts and percentages:")
    for coherence_level, count in coherence_counts_group.items():
        percentage = (count / len(group)) * 100
        print(f"  {coherence_level}: {count} ({percentage:.1f}%)")


ENHANCED ANALYSIS: INDIVIDUAL TYPE COMPONENTS
TYPE COMPONENTS DISTRIBUTION BY CONSTRUCTIVENESS

--- Constructive (n=132) ---
Total type mentions: 152 (from 132 threads)
Individual type component counts and percentages:
  Positive/respectful: 82 (53.9%)
  Argumentative: 26 (17.1%)
  Prersonal stories: 21 (13.8%)
  Snarky/humorous: 10 (6.6%)
  NaN: 9 (5.9%)
  Off-Topic/digression: 3 (2.0%)
  Flamewar: 1 (0.7%)

--- Not constructive (n=52) ---
Total type mentions: 66 (from 52 threads)
Individual type component counts and percentages:
  Flamewar: 15 (22.7%)
  Argumentative: 15 (22.7%)
  Off-Topic/digression: 13 (19.7%)
  Snarky/humorous: 13 (19.7%)
  Prersonal stories: 6 (9.1%)
  NaN: 2 (3.0%)
  Positive/respectful: 2 (3.0%)

--- nan (n=36) ---
Total type mentions: 38 (from 36 threads)
Individual type component counts and percentages:
  NaN: 19 (50.0%)
  Snarky/humorous: 9 (23.7%)
  Off-Topic/digression: 4 (10.5%)
  Positive/respectful: 2 (5.3%)
  Prersonal stories: 2 (5.3%)
  Argumentati